# Population and income features

Add population (WorldPop) and income-proxy layers (Meta's Relative Wealth Index and VIIRS
nighttime lights) at both the station locations and the analysis grid cells from notebook 04
(Section 5.4 / 5.5, Phase F). These are exposure and equity context, not causes of pollution,
so they are kept as separate columns that can be tested for predictive value later.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
from rasterio.enums import Resampling
import xarray as xr
from scipy.spatial import cKDTree
import ee
import dotenv
import yaml

import monitoring_deserts.data.fetch_population as fetch_pop
import monitoring_deserts.data.fetch_earthengine as fetch_ee

## Configuration

In [3]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)

RAW_DATA_DIR = Path(gen_config["data_dir"]["raw"]) / "population"
CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "population"
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

EQUAL_AREA_CRS = gen_config["crs"]["equal_area"]

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())
DAY_START, DAY_END = census_config["period"]["start"], census_config["period"]["end"]

# Countries to download WorldPop / RWI for (ISO 3166-1 alpha-3). Note that BOUNDING_BOX
# also touches countries outside this list (e.g. Afghanistan, Bhutan), so grid cells
# there will have no population/RWI values unless they are added.
COUNTRIES = ["IND", "PAK", "BGD", "NPL", "LKA"]

# Open the analysis df (stations x months, with satellite and meteorology features) from
# notebook 5, and the analysis grid from notebook 4
analysis_df = pd.read_parquet(CLEAN_DATA_DIR.parent / "meteorology" / "analysis_df.parquet")
analysis_grid = gpd.read_parquet(PROCESSED_DATA_DIR / "grid.parquet")

# Initialize Earth Engine client (for VIIRS nighttime lights)
dotenv.load_dotenv()
ee.Initialize(project="monitoring-deserts")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## Step 1 - Download population data

In [5]:
# Download the population data
WORLDPOP_YEAR = 2020
WORLDPOP_RESOLUTION = "1km"
paths = fetch_pop.pull_worldpop_countries(COUNTRIES, RAW_DATA_DIR, year=WORLDPOP_YEAR, resolution=WORLDPOP_RESOLUTION)

[1/5] IND: ind_ppp_2020_1km_Aggregated_UNadj.tif already downloaded, skipping
[2/5] PAK: pak_ppp_2020_1km_Aggregated_UNadj.tif already downloaded, skipping
[3/5] BGD: bgd_ppp_2020_1km_Aggregated_UNadj.tif already downloaded, skipping
[4/5] NPL: npl_ppp_2020_1km_Aggregated_UNadj.tif already downloaded, skipping
[5/5] LKA: lka_ppp_2020_1km_Aggregated_UNadj.tif already downloaded, skipping


## Step 2 - Mosaic the country rasters

In [6]:
# Merge the per-country rasters into one region-wide raster, clipped to the study bounding box
pop_raster = fetch_pop.mosaic_rasters(paths, bbox=BOUNDING_BOX)

# Sanity check: shape, CRS, cell size, and total population inside the bounding box
print(pop_raster.shape, pop_raster.rio.crs, pop_raster.rio.resolution())
print(f"Total population in mosaic: {float(pop_raster.sum()):,.0f}")

(3742, 3898) EPSG:4326 (0.008333333299999999, -0.0083333333)
Total population in mosaic: 1,800,289,280


## Step 3 - Population onto the analysis grid

Population is a count, so source pixels are **summed** into each grid cell (Appendix C.7), not averaged.

In [7]:
# Build an empty raster whose cells are exactly the analysis-grid cells. The grid is regular,
# so its unique centroid longitudes/latitudes give the raster's x/y axes. The centroids are
# rounded because shapely computes them with floating-point noise (~1e-14), which would
# otherwise split each real longitude/latitude into several "unique" values.
grid_lons = np.sort(analysis_grid["centroid_lon"].round(9).unique())
grid_lats = np.sort(analysis_grid["centroid_lat"].round(9).unique())
assert len(grid_lons) * len(grid_lats) == len(analysis_grid), "analysis_grid is not a full regular grid"

template = xr.DataArray(
    np.zeros((len(grid_lats), len(grid_lons)), dtype="float32"),
    coords={"y": grid_lats[::-1], "x": grid_lons},  # rasters are north-up, so y descends
    dims=("y", "x"),
).rio.write_crs(gen_config["crs"]["storage"])

In [8]:
# Resample onto the grid, summing the source pixels that fall in each cell
pop_gridded = pop_raster.rio.reproject_match(template, resampling=Resampling.sum)

# Conservation check: the gridded total should be close to the mosaic's total
src_total = float(pop_raster.sum())
grid_total = float(pop_gridded.sum())
print(f"Mosaic total:  {src_total:,.0f}")
print(f"Gridded total: {grid_total:,.0f}")
print(f"Ratio: {grid_total / src_total:.4f}")

Mosaic total:  1,800,289,280
Gridded total: 1,800,289,152
Ratio: 1.0000


In [9]:
# Read each cell's value back by (row, col), so it is keyed on cell_id rather than row order
col = np.searchsorted(grid_lons, analysis_grid["centroid_lon"].round(9).values)
row = len(grid_lats) - 1 - np.searchsorted(grid_lats, analysis_grid["centroid_lat"].round(9).values)  # y descends

grid_population = pd.DataFrame({
    "cell_id": analysis_grid["cell_id"].values,
    "population": pop_gridded.values[row, col],
})

print(f"{grid_population['population'].isna().sum():,} of {len(grid_population):,} cells have no population data")
grid_population["population"].describe()

279,850 of 434,940 cells have no population data


count    1.550900e+05
mean     1.160803e+04
std      3.293594e+04
min      0.000000e+00
25%      1.565865e+03
50%      5.472563e+03
75%      1.253669e+04
max      2.500372e+06
Name: population, dtype: float64

In [10]:
# Population at each station: the value of the grid cell that contains it, so the training
# features match what the model sees on the grid at prediction time
CELL_SIZE = gen_config["grid_cell_size_deg"]

# One row per station (analysis_df has one row per station-month)
unique_stations = analysis_df[["station_id", "longitude", "latitude"]].drop_duplicates("station_id")

st_col, st_lat_idx = fetch_pop.locate_grid_cells(unique_stations["longitude"], unique_stations["latitude"],
                                                 grid_lons, grid_lats, CELL_SIZE)
st_row = len(grid_lats) - 1 - st_lat_idx  # y descends

# cell_id of each raster position, looked up from the grid itself rather than assumed from its ordering
cell_id_raster = np.empty((len(grid_lats), len(grid_lons)), dtype=analysis_grid["cell_id"].dtype)
cell_id_raster[row, col] = analysis_grid["cell_id"].values

station_population = pd.DataFrame({
    "station_id": unique_stations["station_id"].values,
    "cell_id": cell_id_raster[st_row, st_col],
    "population": pop_gridded.values[st_row, st_col],
})

print(f"Stations without population data: {station_population['population'].isna().sum()} of {len(station_population)}")

Stations without population data: 0 of 344


## Step 4 - Relative Wealth Index (income proxy)

Nearest RWI point for every station and grid cell (Section 5.5). Locations with no RWI point within the cutoff get NaN, not a far-away value.

In [11]:
# Download the RWI files (India and Pakistan share one) and read them into one table of wealth points
rwi_paths = fetch_pop.pull_rwi(COUNTRIES, RAW_DATA_DIR / "rwi")
rwi = fetch_pop.load_rwi(rwi_paths)

print(f"{len(rwi):,} RWI points from {len(set(rwi_paths.values()))} files")
rwi["rwi"].describe()

[1/4] ind_pak_relative_wealth_index.csv: ind_pak_relative_wealth_index.csv already downloaded, skipping
[2/4] bgd_relative_wealth_index.csv: bgd_relative_wealth_index.csv already downloaded, skipping
[3/4] npl_relative_wealth_index.csv: npl_relative_wealth_index.csv already downloaded, skipping
[4/4] lka_relative_wealth_index.csv: lka_relative_wealth_index.csv already downloaded, skipping
646,933 RWI points from 4 files


count    646933.000000
mean         -0.143886
std           0.413224
min          -1.578000
25%          -0.438000
50%          -0.207000
75%           0.092000
max           2.162000
Name: rwi, dtype: float64

In [12]:
# Nearest RWI point for every station and grid cell; NaN if none lies within the cutoff
RWI_MAX_DISTANCE_KM = 5.0

# One row per station (analysis_df has one row per station-month)
unique_stations = analysis_df[["station_id", "longitude", "latitude"]].drop_duplicates("station_id")

station_rwi = fetch_pop.nearest_rwi(unique_stations["longitude"], unique_stations["latitude"],
                                    rwi, max_distance_km=RWI_MAX_DISTANCE_KM)
station_rwi.insert(0, "station_id", unique_stations["station_id"].values)

grid_rwi = fetch_pop.nearest_rwi(analysis_grid["centroid_lon"], analysis_grid["centroid_lat"],
                                 rwi, max_distance_km=RWI_MAX_DISTANCE_KM)
grid_rwi.insert(0, "cell_id", analysis_grid["cell_id"].values)

print(f"Stations without RWI: {station_rwi['rwi'].isna().sum():,} of {len(station_rwi):,}")
print(f"Grid cells without RWI: {grid_rwi['rwi'].isna().sum():,} of {len(grid_rwi):,}")

Stations without RWI: 0 of 344
Grid cells without RWI: 297,598 of 434,940


## Step 5 - VIIRS nighttime lights

Annual composite radiance, a static (no month) activity/wealth proxy. Stations get the mean over a buffer, like the satellite features in notebook 03. Grid cells get the mean of the VIIRS pixels inside each cell, downloaded in tiles, because `reduceRegions` over 435,000 cells doesn't scale.

In [13]:
VIIRS_COLLECTION = "NOAA/VIIRS/DNB/ANNUAL_V22"
VIIRS_BAND = "average"
VIIRS_YEAR = 2025          # latest annual composite; overlaps most of the study period
BUFFER_RADIUS = 2000       # m, same as the satellite features (notebook 03)
REDUCER_SCALE = 500        # m, VIIRS DNB's native resolution is ~460 m

viirs = ee.ImageCollection(VIIRS_COLLECTION).select(VIIRS_BAND)
year_window = [(f"{VIIRS_YEAR}-01-01", f"{VIIRS_YEAR + 1}-01-01")]

# Stations: mean (and spread) over a buffer around each station. pull_monthly_stats takes a list
# of windows, so a single annual window gives one row per station.
station_fc = fetch_ee.stations_to_feature_collection(unique_stations, radius_m=BUFFER_RADIUS)
viirs_df = fetch_ee.pull_monthly_stats(viirs, station_fc, year_window, "viirs", scale=REDUCER_SCALE)[0]

station_viirs = viirs_df[["station_id", "mean", "stdDev"]].rename(columns={"mean": "viirs", "stdDev": "viirs_sd"})
station_viirs["station_id"] = station_viirs["station_id"].astype(unique_stations["station_id"].dtype)

print(f"Stations without VIIRS: {station_viirs['viirs'].isna().sum()} of {len(station_viirs)}")

Stations without VIIRS: 0 of 344


In [14]:
# Grid cells: mean of the VIIRS pixels inside each cell, downloaded in tiles. .first() keeps the
# image's native projection, which the resampling needs (a .mean() composite loses it).
viirs_image = viirs.filterDate(*year_window[0]).first()
viirs_grid = fetch_ee.download_image_on_grid(viirs_image, grid_lons, grid_lats, CELL_SIZE)

grid_viirs = pd.DataFrame({
    "cell_id": analysis_grid["cell_id"].values,
    "viirs": viirs_grid[row, col],
})

print(f"Grid cells without VIIRS: {grid_viirs['viirs'].isna().sum():,} of {len(grid_viirs):,}")
grid_viirs["viirs"].describe()

Grid tiles:   0%|          | 0/16 [00:00<?, ?it/s]

Grid cells without VIIRS: 0 of 434,940


count    434940.000000
mean          0.680344
std           2.110200
min          -0.374041
25%           0.307613
50%           0.393022
75%           0.535742
max         815.336609
Name: viirs, dtype: float64

## Step 6 - Merge, save and record metadata

Station features are static (no month), so they merge onto `analysis_df` on `station_id` only. Grid features are keyed on `cell_id`.

In [15]:
# Station features merge onto analysis_df on station_id only (they have no month), keeping every row
station_features = (station_population
                    .merge(station_rwi, on="station_id")
                    .merge(station_viirs, on="station_id"))
assert station_features["station_id"].is_unique

new_analysis_df = analysis_df.merge(station_features, on="station_id", how="left")
assert len(new_analysis_df) == len(analysis_df), "merge changed the number of rows"
new_analysis_df.to_parquet(CLEAN_DATA_DIR / "analysis_df.parquet")

# Grid features, one row per cell
grid_features = (analysis_grid[["cell_id", "centroid_lon", "centroid_lat"]]
                 .merge(grid_population, on="cell_id")
                 .merge(grid_rwi, on="cell_id")
                 .merge(grid_viirs, on="cell_id"))
assert len(grid_features) == len(analysis_grid), "merge changed the number of cells"
grid_features.to_parquet(CLEAN_DATA_DIR / "grid_features.parquet")

print(new_analysis_df.shape, grid_features.shape)
new_analysis_df.head()

(5504, 30) (434940, 8)


,month,station_id,longitude,latitude,no2,no2_sd,no2_units,aai,aai_sd,aai_units,pm25,pm25_sd,pm25_units,pm25_nan_reason,total_precipitation,temperature,dew_point,u_wind,v_wind,boundary_layer_height,surface_pressure,relative_humidity,wind_magnitude,cell_id,population,rwi,rwi_error,rwi_distance_km,viirs,viirs_sd
0,2025-03,50,77.131023,28.674045,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless,78.702397,25.853544,ugm3,none,0.000355,296.169999,285.891946,1.188648,-1.215477,502.741195,98572.145820,52.448721,1.701274,220572,902154.000000,1.279,0.484,0.860622,47.109492,7.478950
1,2025-03,235,77.316032,28.646835,0.000134,0.000005,mol/m2,0.236869,0.021313,dimensionless,81.005496,31.516710,ugm3,none,0.000353,296.211719,285.973674,1.306716,-1.258353,503.958572,98598.230938,52.584996,1.815828,222548,893627.625000,1.398,0.499,0.711349,57.317502,7.121097
2,2025-03,860,78.005981,27.198658,0.000058,0.000001,mol/m2,0.331405,0.024480,dimensionless,57.440896,17.897299,ugm3,none,0.000115,297.523394,286.185682,0.837137,-0.941065,481.071154,99138.629691,49.242951,1.259937,231745,398128.656250,1.568,0.512,1.115314,44.088553,7.805645
3,2025-03,2456,80.891736,26.833997,0.000051,0.000001,mol/m2,0.428689,0.022733,dimensionless,58.161202,20.561508,ugm3,none,0.000024,297.821274,285.790882,2.046100,-0.949337,559.689115,99596.172373,47.142067,2.256312,269960,334333.718750,1.569,0.507,0.060884,38.699784,8.695076
4,2025-03,5408,80.518167,16.515083,0.000067,0.000004,mol/m2,-0.063205,0.019414,dimensionless,24.145945,15.724409,ugm3,none,0.000036,303.195387,294.690451,-0.471233,1.819787,764.035770,100507.773666,60.433367,1.880140,264482,12655.417969,0.127,0.464,0.591219,5.755284,3.017830


In [16]:
# Notebook metadata
metadata = {
    "worldpop": {
        "year": WORLDPOP_YEAR,
        "resolution": WORLDPOP_RESOLUTION,
        "countries": COUNTRIES,
        "mosaic_total": int(src_total),
        "gridded_total": int(grid_total),
        "n_cells_with_population": int(grid_population["population"].notna().sum()),
        "n_stations_without_population": int(station_population["population"].isna().sum()),
    },
    "rwi": {
        "files": sorted({p.name for p in rwi_paths.values()}),
        "n_points": len(rwi),
        "max_distance_km": RWI_MAX_DISTANCE_KM,
        "n_stations_without_rwi": int(station_rwi["rwi"].isna().sum()),
        "n_cells_with_rwi": int(grid_rwi["rwi"].notna().sum()),
    },
    "viirs": {
        "collection": VIIRS_COLLECTION,
        "band": VIIRS_BAND,
        "year": VIIRS_YEAR,
        "station_buffer_radius": BUFFER_RADIUS,
        "reducer_scale_m": REDUCER_SCALE,
        "grid_method": "mean of source pixels per cell, tiled getDownloadURL",
        "n_stations_without_viirs": int(station_viirs["viirs"].isna().sum()),
        "n_cells_without_viirs": int(grid_viirs["viirs"].isna().sum()),
    },
    "n_stations": int(new_analysis_df["station_id"].nunique()),
    "n_data_points": len(new_analysis_df),
    "n_grid_cells": len(grid_features),
}

with open("../configs/population_income_metadata.yaml", "w") as f:
    yaml.safe_dump(metadata, f, sort_keys=False)